<a href="https://colab.research.google.com/github/craljimenez/An-lisis_y_Visualizacion_de_Datos_para_la_Toma_de_Decisiones/blob/main/clases/semana04-clase10-merge-concatenacion-integracion/notebooks/lab-merge-concatenacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratorio: Merge, concatenación e integración de fuentes
### Semana 5 · Clase 10
**Curso:** Análisis y Visualización de Datos para la Toma de Decisiones (Esumer)

Hasta ahora trabajamos con una sola tabla. En la práctica, la información de un negocio vive repartida: los socios en un archivo, los planes en un catálogo, las sedes en otro, las nuevas inscripciones en un tercero. Integrar esas fuentes correctamente es una de las tareas que más errores produce en un análisis, porque un paso mal hecho cambia silenciosamente el número de filas o el valor de una columna.

Hoy aprendemos dos herramientas: **`merge`** para unir tablas por una llave (como un `JOIN` de SQL) y **`concat`** para apilar tablas (una debajo de otra, o una al lado de otra). Y, sobre todo, aprendemos a **verificar** cada paso.

In [ ]:
from google.colab import files

uploaded = files.upload()

Selecciona los **cuatro** archivos de la carpeta `datos/` de esta clase, en el repositorio del curso: `membresias_gimnasio_limpio.csv`, `planes.csv`, `sedes.csv` y `nuevas_inscripciones.csv`.

In [ ]:
import pandas as pd

socios = pd.read_csv("membresias_gimnasio_limpio.csv")
planes = pd.read_csv("planes.csv")
sedes = pd.read_csv("sedes.csv")
nuevas = pd.read_csv("nuevas_inscripciones.csv")

for nombre, tabla in [("socios", socios), ("planes", planes), ("sedes", sedes), ("nuevas", nuevas)]:
    print(f"{nombre:8s} {tabla.shape}")

ModuleNotFoundError: No module named 'pandas'

## 1. Antes de unir: qué es una llave

Una **llave** es la columna que identifica a qué fila de una tabla corresponde una fila de la otra. En nuestras fuentes:

| Fuente | Qué contiene | Llave que la conecta |
|---|---|---|
| `socios` | una fila por socio | `plan` (con `planes`) y `ciudad` (con `sedes`) |
| `planes` | una fila por plan | `plan` |
| `sedes` | una fila por ciudad | `ciudad` |
| `nuevas` | nuevas inscripciones, mismo esquema que `socios` | `id_socio` (para detectar repetidos) |

Antes de unir, revisa tres cosas de cada llave: que sea **única** en la tabla que describe (un plan no debería aparecer dos veces en `planes`), que tenga el **mismo tipo** en ambas tablas, y qué valores **no coinciden** entre ellas.

In [ ]:
print("planes.plan es única:", planes["plan"].is_unique)
print("sedes.ciudad es única:", sedes["ciudad"].is_unique)
print("socios.id_socio es única:", socios["id_socio"].is_unique)
print()
print("tipos de la llave:", socios["plan"].dtype, "/", planes["plan"].dtype)
print()
print("planes que ningún socio tiene:", sorted(set(planes["plan"]) - set(socios["plan"])))
print("ciudades de socios sin sede:", sorted(set(socios["ciudad"]) - set(sedes["ciudad"])))

Ya sabemos dos cosas importantes: `Semestral` existe en el catálogo pero ningún socio lo tiene, y `Itagüí` aparece en los socios pero la sede lo tiene escrito sin tilde (`Itagui`). El segundo caso es un problema de consistencia que vamos a resolver antes de unir, en la Sección 4.

## 2. `merge`: unir por una llave

`merge` une dos tablas buscando filas con el mismo valor de la llave. Lo más importante es el parámetro `how`, que decide qué filas se conservan:

| `how` | Qué conserva | Equivalente en SQL |
|---|---|---|
| `"inner"` | solo las llaves que existen en **ambas** tablas | `INNER JOIN` |
| `"left"` | todas las filas de la **izquierda**, con o sin pareja | `LEFT JOIN` |
| `"right"` | todas las filas de la **derecha**, con o sin pareja | `RIGHT JOIN` |
| `"outer"` | **todas** las filas de ambas tablas | `FULL OUTER JOIN` |

Compara cuántas filas da cada uno al unir los socios con el catálogo de planes:

In [ ]:
resultados = {}
for how in ["inner", "left", "right", "outer"]:
    resultados[how] = socios.merge(planes, on="plan", how=how).shape[0]
pd.Series(resultados, name="filas")

`inner` y `left` dan 50 filas: todos los socios tienen un plan del catálogo. `right` y `outer` dan 51 porque conservan `Semestral`, el plan sin socios, con los datos de socios en blanco. Por eso el tipo de unión no es un detalle técnico: **decide qué entidades sobreviven a la integración**.

En la mayoría de los análisis de socios, quieres `left`: partir de la tabla principal y agregarle información, sin perder a nadie.

Los tipos de unión son más fáciles de recordar con un diagrama de Venn. Imagina cada tabla como un círculo: el área que se queda en verde es lo que el `merge` conserva, y el gris lo que descarta. Los números son las filas reales de nuestras tablas: cuántos socios tienen plan en el catálogo, cuántos socios no lo tienen, y cuántos planes no tienen socios.

In [1]:
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

COLOR_CONSERVA = "#0CB3B6"
COLOR_DESCARTA = "#E8E8E8"
CENTRO_IZQ, CENTRO_DER, RADIO = -0.55, 0.55, 1.0

llaves_planes = set(planes["plan"])
llaves_socios = set(socios["plan"])
n_ambas = int(socios["plan"].isin(llaves_planes).sum())
n_solo_izq = int((~socios["plan"].isin(llaves_planes)).sum())
n_solo_der = int((~planes["plan"].isin(llaves_socios)).sum())

conserva = {
    "inner": {"ambas"},
    "left": {"ambas", "izq"},
    "right": {"ambas", "der"},
    "outer": {"ambas", "izq", "der"},
}

def color(how, region):
    return COLOR_CONSERVA if region in conserva[how] else COLOR_DESCARTA

def dibujar(ax, how):
    ax.add_patch(Circle((CENTRO_IZQ, 0), RADIO, color=color(how, "izq"), zorder=1))
    ax.add_patch(Circle((CENTRO_DER, 0), RADIO, color=color(how, "der"), zorder=2))
    interseccion = Circle((CENTRO_DER, 0), RADIO, color=color(how, "ambas"), zorder=3)
    interseccion.set_clip_path(Circle((CENTRO_IZQ, 0), RADIO, transform=ax.transData))
    ax.add_patch(interseccion)
    for cx in (CENTRO_IZQ, CENTRO_DER):
        ax.add_patch(Circle((cx, 0), RADIO, fill=False, edgecolor="#262E66", linewidth=1.6, zorder=4))
    ax.text(-1.25, 0, str(n_solo_izq), ha="center", va="center", fontsize=13, zorder=5)
    ax.text(0, 0, str(n_ambas), ha="center", va="center", fontsize=13, zorder=5)
    ax.text(1.25, 0, str(n_solo_der), ha="center", va="center", fontsize=13, zorder=5)
    ax.text(CENTRO_IZQ, 1.2, "socios", ha="center", fontsize=10, color="#262E66")
    ax.text(CENTRO_DER, 1.2, "planes", ha="center", fontsize=10, color="#262E66")
    filas = len(socios.merge(planes, on="plan", how=how))
    ax.set_title(f"how='{how}': {filas} filas", fontsize=12)
    ax.set_xlim(-2, 2)
    ax.set_ylim(-1.4, 1.5)
    ax.set_aspect("equal")
    ax.axis("off")

fig, axes = plt.subplots(1, 4, figsize=(16, 4.6))
for ax, how in zip(axes, ["inner", "left", "right", "outer"]):
    dibujar(ax, how)
fig.suptitle("Qué filas conserva cada tipo de unión (verde: se conserva; gris: se descarta)", fontsize=13)
fig.tight_layout()
plt.show()

NameError: name 'planes' is not defined

### 2.1 Saber qué se emparejó: `indicator=True`

Con `indicator=True`, pandas agrega una columna `_merge` que dice de dónde vino cada fila. Es la forma más directa de **detectar registros no emparejados**:

In [ ]:
union = planes.merge(socios, on="plan", how="left", indicator=True)
union["_merge"].value_counts()

In [ ]:
union.loc[union["_merge"] == "left_only", ["plan", "beneficio"]]

`left_only` son las filas de la izquierda (`planes`) que no encontraron pareja: aquí, `Semestral`. Esa es exactamente la lista que necesitas revisar antes de dar por buena una integración: ¿es un plan sin ventas todavía, o es un error de escritura en la llave?

Recuerda también que, si las columnas se llaman distinto en cada tabla, usas `left_on` y `right_on` en lugar de `on`.

## 3. Consistencia antes de unir: normalizar la llave

Las llaves de texto son frágiles. `Itagüí` y `Itagui` son, para una persona, la misma ciudad; para pandas son dos valores distintos. Una integración que no normaliza primero deja registros sin pareja sin dar ningún error.

Primero, mide cuántos socios se quedarían sin sede:

In [ ]:
sin_sede_antes = socios.merge(sedes, on="ciudad", how="left", indicator=True)
print(sin_sede_antes["_merge"].value_counts())
print()
print(sin_sede_antes.loc[sin_sede_antes["_merge"] == "left_only", "ciudad"].value_counts())

Ahora corrige la llave en la tabla de sedes (no en los socios, para no tocar el dato original) y vuelve a medir:

In [ ]:
sedes_normalizadas = sedes.copy()
sedes_normalizadas["ciudad"] = sedes_normalizadas["ciudad"].replace({"Itagui": "Itagüí"})

sin_sede_despues = socios.merge(sedes_normalizadas, on="ciudad", how="left", indicator=True)
sin_sede_despues["_merge"].value_counts()

Ya no queda ningún socio sin sede. Regla práctica: **antes de unir, normaliza la llave** (espacios, mayúsculas, tildes) y **después de unir, cuenta los `left_only` y los `right_only`**. Si el número no es cero, cada caso debe quedar documentado: no lo ignores.

Un chequeo más fino: cuando el catálogo trae un valor (como el precio), compara la columna del catálogo con la que ya tenía la tabla principal. Si hay diferencias, algo está desactualizado en alguna de las dos fuentes:

In [ ]:
chequeo = socios.merge(planes[["plan", "precio_catalogo_cop"]], on="plan", how="left")
discrepancias = chequeo[chequeo["precio_plan_cop"] != chequeo["precio_catalogo_cop"]]
print("socios con precio distinto al catálogo:", len(discrepancias))

## 4. Cardinalidad: ¿cuántas parejas puede tener cada fila?

La **cardinalidad** describe la relación entre las dos tablas:

| Relación | Ejemplo | Qué esperas al unir |
|---|---|---|
| uno a uno (`1:1`) | un socio tiene un expediente | mismo número de filas |
| muchos a uno (`m:1`) | muchos socios pertenecen a una ciudad | mismo número de filas que la tabla de muchos |
| muchos a muchos (`m:m`) | llave repetida en ambas tablas | **explosión de filas** |

El caso más común en un análisis es `m:1`: muchos socios, una sola fila por ciudad en `sedes`. Si la tabla "de búsqueda" tiene una llave repetida, el `merge` no falla: multiplica filas en silencio.

Veamos qué pasa si `sedes` tuviera dos filas para Medellín:

In [ ]:
sedes_con_error = pd.concat(
    [sedes_normalizadas, pd.DataFrame([["Medellín", "Sede Centro", "Andrés Mora"]], columns=sedes.columns)],
    ignore_index=True,
)
print("filas de socios:", len(socios))
print("filas después del merge sin validar:", len(socios.merge(sedes_con_error, on="ciudad", how="left")))

De 50 socios pasamos a 72 filas: los 22 socios de Medellín se duplicaron. Ningún error, ningún aviso, y cualquier conteo posterior estaría inflado. Para evitarlo, pide a pandas que **valide la cardinalidad** con el parámetro `validate`:

In [ ]:
try:
    socios.merge(sedes_con_error, on="ciudad", how="left", validate="many_to_one")
except pd.errors.MergeError as error:
    print("MergeError:", error)

Con `validate="many_to_one"` el error aparece de inmediato, en el momento de la integración. Y con la tabla limpia la misma validación pasa sin problema:

In [ ]:
union_validada = socios.merge(sedes_normalizadas, on="ciudad", how="left", validate="many_to_one")
print("filas:", len(union_validada))

Usa `validate` siempre que hagas un `merge`: documenta la relación que esperas y deja que pandas la compruebe. Las opciones son `"one_to_one"`, `"one_to_many"`, `"many_to_one"` y `"many_to_many"`.

## 5. `concat`: apilar tablas

`concat` no busca llaves: **apila** tablas. Lo más común es apilar filas (`axis=0`), por ejemplo cuando las nuevas inscripciones llegan en un archivo aparte con el mismo esquema:

In [ ]:
apilada = pd.concat([socios, nuevas], ignore_index=True)
print("filas antes:", len(socios), "+", len(nuevas), "=", len(apilada))

`ignore_index=True` renumera el índice desde cero; sin esa opción, las filas conservan su índice original y pueden aparecer números repetidos.

Antes de dar por buena la tabla apilada, revisa que la llave sigue siendo única:

In [ ]:
repetidos = apilada[apilada.duplicated("id_socio", keep=False)]
repetidos[["id_socio", "nombre", "ciudad"]].sort_values("id_socio")

El `id_socio` 7 existe dos veces, con dos personas distintas: `Alejandra Vélez Martínez` (ya en la base) y `Carlos Andrés Peña` (la nueva inscripción). No es una duplicación que se pueda borrar sin más: alguien debe decidir cuál registro es el correcto, o si el nuevo debe recibir otro identificador. Por eso el paso correcto es **detectarlo y decidir**, no eliminar una de las filas automáticamente.

Si quieres que pandas falle al encontrar un índice repetido, usa `verify_integrity=True`. Aplica sobre el índice, así que primero fija `id_socio` como índice (lo viste en la Clase 7):

In [ ]:
try:
    pd.concat([socios.set_index("id_socio"), nuevas.set_index("id_socio")], verify_integrity=True)
except ValueError as error:
    print("ValueError:", error)

Un detalle sobre el esquema: si las dos tablas tienen columnas distintas, `concat` las junta todas y deja en blanco (`NaN`) los valores que una tabla no tiene. Eso no es un error, pero conviene revisar el resultado: un `NaN` en una columna que sí debería existir es una señal de que el esquema no coincide.

## 6. `concat` horizontal: pegar columnas

Con `axis=1`, `concat` pega tablas **una al lado de la otra**, y lo hace alineando por el **índice**, no por el orden de las filas. Si los índices no coinciden, aparecen huecos:

In [ ]:
izquierda = socios[["id_socio", "nombre"]].head(3)
derecha = socios[["ciudad"]].iloc[[1, 2, 3]]
pd.concat([izquierda, derecha], axis=1)

Las filas 0 y 3 quedaron incompletas: el índice de `derecha` empieza en 1, así que no se alineó con el de `izquierda`. La solución es fijar una llave explícita antes de pegar (aquí `id_socio`):

In [ ]:
pd.concat(
    [izquierda.set_index("id_socio"), socios.set_index("id_socio")[["ciudad"]].head(3)],
    axis=1,
)

Para pegar columnas, la forma más segura suele ser `merge` por la llave. `concat` horizontal solo es razonable cuando sabes con certeza que las dos tablas tienen el mismo índice y el mismo orden.

## 7. Validar la calidad de la integración

Una integración no está terminada hasta que la verificas. Estos chequeos son un mínimo que conviene correr siempre después de unir:

1. ¿Cuántas filas tenía la tabla principal, y cuántas tiene el resultado?
2. ¿La llave sigue siendo única donde debe serlo?
3. ¿Cuántos registros quedaron sin pareja?
4. ¿Los valores que vienen del catálogo coinciden con los de la tabla principal?

Armemos la tabla maestra de socios con sede y plan, y pasemos esos cuatro chequeos:

In [ ]:
maestra = (
    socios
    .merge(planes[["plan", "meses", "precio_catalogo_cop"]], on="plan", how="left", validate="many_to_one")
    .merge(sedes_normalizadas, on="ciudad", how="left", validate="many_to_one")
)

chequeos = {
    "filas de socios (entrada)": len(socios),
    "filas de la maestra (salida)": len(maestra),
    "id_socio único en la maestra": maestra["id_socio"].is_unique,
    "socios sin plan en el catálogo": int(maestra["meses"].isna().sum()),
    "socios sin sede": int(maestra["sede"].isna().sum()),
    "precio distinto al catálogo": int((maestra["precio_plan_cop"] != maestra["precio_catalogo_cop"]).sum()),
}
pd.Series(chequeos, name="resultado")

Lee el resultado con criterio: la maestra debe tener **las mismas 50 filas** que la entrada (no se perdió ni se duplicó ningún socio), la llave es única, no queda nadie sin plan ni sin sede, y el precio coincide con el catálogo. Si algo de esto falla, no sigas con el análisis hasta entender por qué.

## 8. Reto: integra y valida

Tu entregable tiene tres partes:

1. **Integra al menos dos fuentes.** Puedes usar las del laboratorio (por ejemplo, agregar las nuevas inscripciones a los socios y después unirles la sede), o bien integrar tu propio dataset de la Clase 7 y 8 con un catálogo que tú crees (una tabla pequeña con una llave en común).
2. **Usa `validate`** en cada `merge`, y explica en una celda de markdown qué relación esperabas (`1:1`, `m:1` o `m:m`).
3. **Documenta la validación**: una tabla con los chequeos de la Sección 7 y **una decisión** tomada sobre algún registro problemático (un `id_socio` repetido, una llave sin pareja, un valor distinto al catálogo). Explica por qué la tomaste.

In [ ]:
# TODO: integra las fuentes en una sola tabla (usa merge o concat según corresponda)


In [ ]:
# TODO: valida la integración con los chequeos de la Sección 7


**Relación que esperaba en cada unión:** *(1:1, m:1 o m:m, y por qué)*

**Registros problemáticos encontrados:** *(cuántos, cuáles, y cómo los detecté)*

**Decisión tomada:** *(qué hice con ellos y por qué, en vez de borrarlos o ignorarlos sin más)*

## Cierre

Hoy integraste fuentes separadas. Recuerda el orden: **revisa las llaves** antes de unir (unicidad, tipo, valores que no coinciden), **elige el tipo de unión** según qué filas debe conservar tu análisis, **valida la cardinalidad** con `validate`, y **verifica el resultado** contando lo que quedó sin pareja.

Guarda una copia de este notebook en tu Google Drive antes de cerrar Colab.

En la **Clase 11: Principios de visualización y comunicación de datos** vas a tomar la tabla maestra que construiste hoy (o la que resultó de la Clase 9) y empezar a preguntarte cómo mostrarla para que un público la entienda sin ayuda.